# 导入包

In [1]:
import sys
import importlib

# 1. 将你的本地存放包的绝对路径加入到系统搜索路径中
factor_path = r"C:\Users\CtaBackup\Desktop\opt\factor"
if factor_path not in sys.path:
    sys.path.append(factor_path)

# 2. 正常导入这些模块（如果内核刚重启，这是第一次导入；如果之前导入过，这里走缓存）
import fmb_predict_combined
import my_backtest
import quant_eval
import cluster_toolkit

# 3. 核心步骤：强制重新加载模块，覆盖旧的缓存代码
importlib.reload(fmb_predict_combined)
importlib.reload(my_backtest)
importlib.reload(quant_eval)
importlib.reload(cluster_toolkit)

print("✅ 本地模块已强制重新加载完毕，现在使用的是最新代码！")

✅ 本地模块已强制重新加载完毕，现在使用的是最新代码！


# 数据
- 主力合约结算价（名称：算主力合约到期天数）
- 成交额
- 持仓量


In [2]:
import pandas as pd
import os
import glob

# 设置路径
folder_path = r'C:\Users\CtaBackup\Desktop\py_data\day1_2'

# 方法1: 使用 glob 读取所有 CSV 文件
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# 定义标准列名
standard_columns = [
    'date',                # 对应 day1(:, 1)
    'open',                # 对应 day1(:, 2)
    'high',                # 对应 day1(:, 3)
    'low',                 # 对应 day1(:, 4)
    'open',               # 对应 day1(:, 5)
    'volume',              # 对应 day1(:, 6)
    'turnover',            # 对应 day1(:, 7)
    'open_interest',       # 对应 day1(:, 8)
    'volume_total',        # 对应 day1(:, 9)
    'open_interest_total', # 对应 day1(:, 10)
    'carry',               # 对应 day1(:, 11)
    'open_raw',           # 对应 day1(:, 12)
    'carry2',              # 对应 day1(:, 13)
    'receipt',             # 对应 day1(:, 14)
    'limit_up',            # 对应 day1(:, 15)
    'limit_down',          # 对应 day1(:, 16)
    'hycc_long_top5',      # 对应 day1(:, 17)
    'hycc_short_top5',     # 对应 day1(:, 18)
    'hycc_volume_top5',    # 对应 day1(:, 19)
    'spread1'              # 对应 day1(:, 20)
]
# 创建字典存储所有数据
data_dict = {}
for file in csv_files:
    filename = os.path.basename(file).replace('.csv', '')
    
    # 添加错误处理，跳过空文件
    try:
        df = pd.read_csv(file)
        
        # 检查DataFrame是否为空
        if df.empty or len(df.columns) == 0:
            print(f"跳过空文件: {filename}")
            continue
        
        # 根据实际列数调整列名
        actual_cols = min(len(df.columns), len(standard_columns))
        new_columns = standard_columns[:actual_cols] + df.columns[actual_cols:].tolist()
        
        # 重命名列
        df.columns = new_columns
        data_dict[filename] = df
        
        print(f"已加载: {filename}, 形状: {df.shape}, 列名: {list(df.columns)}")
    
    except pd.errors.EmptyDataError:
        print(f"跳过空文件: {filename}")
        continue
    except Exception as e:
        print(f"读取文件 {filename} 时出错: {str(e)}")
        continue

print(f"总共加载了 {len(data_dict)} 个CSV文件")

已加载: a, 形状: (2369, 20), 列名: ['date', 'open', 'high', 'low', 'open', 'volume', 'turnover', 'open_interest', 'volume_total', 'open_interest_total', 'carry', 'open_raw', 'carry2', 'receipt', 'limit_up', 'limit_down', 'hycc_long_top5', 'hycc_short_top5', 'hycc_volume_top5', 'spread1']
已加载: ag, 形状: (2369, 20), 列名: ['date', 'open', 'high', 'low', 'open', 'volume', 'turnover', 'open_interest', 'volume_total', 'open_interest_total', 'carry', 'open_raw', 'carry2', 'receipt', 'limit_up', 'limit_down', 'hycc_long_top5', 'hycc_short_top5', 'hycc_volume_top5', 'spread1']
已加载: al, 形状: (2369, 20), 列名: ['date', 'open', 'high', 'low', 'open', 'volume', 'turnover', 'open_interest', 'volume_total', 'open_interest_total', 'carry', 'open_raw', 'carry2', 'receipt', 'limit_up', 'limit_down', 'hycc_long_top5', 'hycc_short_top5', 'hycc_volume_top5', 'spread1']
已加载: ao, 形状: (556, 20), 列名: ['date', 'open', 'high', 'low', 'open', 'volume', 'turnover', 'open_interest', 'volume_total', 'open_interest_total', 'carry

In [3]:
import os
import pandas as pd

# 设置输出文件夹路径
output_folder = r'C:\Users\CtaBackup\Desktop\opt\data'
os.makedirs(output_folder, exist_ok=True)

# =========================================================
# 1. 构建 open 面板：第一列为日期，后续每列为对应品种的 open 值
# =========================================================
all_dates = sorted(set().union(*[set(df.iloc[:,0].values) for df in data_dict.values()]))
open_df = pd.DataFrame({'date': all_dates})

for name, df in data_dict.items():
    # 🌟 修复重名列报错：如果存在多个 'open' 列，强行只取第一个
    open_data = df.loc[:, 'open']
    if isinstance(open_data, pd.DataFrame):
        open_col = open_data.iloc[:, 0].values
    else:
        open_col = open_data.values
        
    series = pd.Series(open_col, index=df.iloc[:,0].values)
    open_df[name] = open_df['date'].map(series)

# 将 date 设置为索引
open_panel = open_df.set_index('date')
open_panel.to_csv(os.path.join(output_folder, 'historical_prices.csv'), index=True)

# 计算 returns：强制数值化，然后按行日期计算 pct_change（即日度/观测周期收益）
returns = open_panel.apply(pd.to_numeric, errors='coerce').pct_change()
# 保存 returns 到 CSV
returns.to_csv(os.path.join(output_folder, 'historical_returns.csv'), index=True)


# =========================================================
# 2. 构建 volume 面板：第一列为日期，后续每列为对应品种的 volume 值
# =========================================================
volume_df = pd.DataFrame({'date': all_dates})

for name, df in data_dict.items():
    # 🌟 修复重名列报错：同样防范 'volume' 出现重名列报错
    volume_data = df.loc[:, 'volume']
    if isinstance(volume_data, pd.DataFrame):
        volume_col = volume_data.iloc[:, 0].values
    else:
        volume_col = volume_data.values
        
    series = pd.Series(volume_col, index=df.iloc[:,0].values)
    volume_df[name] = volume_df['date'].map(series)

# 将 date 设置为索引
volume_panel = volume_df.set_index('date')
# volume_panel.to_csv(os.path.join(output_folder, 'historical_volumes.csv'), index=True)

print(f"✅ Open 收益率面板 和 Volume 成交量面板 提取完毕！(已成功解决重复列名报错)")

✅ Open 收益率面板 和 Volume 成交量面板 提取完毕！(已成功解决重复列名报错)


In [4]:
import pandas as pd

factors = [
    'open', 'volume', 'open_interest', 'receipt', 'limit_up', 'limit_down', 
    'hycc_long_top5', 'hycc_short_top5', 'hycc_volume_top5', 'spread1'
]

# 1. 获取所有唯一的日期，作为对齐基准
all_dates = sorted(set().union(*[set(df.iloc[:,0].values) for df in data_dict.values()]))

# 2. 直接把日期作为无名索引传入，根本不创建 'date' 列
temp_dfs = {factor: pd.DataFrame(index=all_dates) for factor in factors}

# 3. 遍历数据填入
for name, df in data_dict.items():
    col_map = {str(c).strip().lower(): c for c in df.columns}
    
    for factor in factors:
        factor_lower = factor.lower()
        if factor_lower not in col_map:
            continue
            
        actual_col = col_map[factor_lower]
        
        # =========================================================
        # 🌟 修复重名列报错：如果某个因子列在表里有两列同名的，强行取第一列！
        # =========================================================
        raw_data = df.loc[:, actual_col]
        if isinstance(raw_data, pd.DataFrame):
            val_array = raw_data.iloc[:, 0].values  # 如果是多维矩阵，强行切片取第一列
        else:
            val_array = raw_data.values             # 如果是单列，直接取值
            
        # 提取当前品种的数据，并以它的第一列（日期）为索引
        series = pd.Series(val_array, index=df.iloc[:,0].values)
        
        # 直接按照索引对齐赋值，列名为品种名
        temp_dfs[factor][name] = series

# 4. 提取最常用的几个面板方便核对检查
open_df = temp_dfs['open']
volume_df = temp_dfs['volume']

print(f"✅ 10个量价与基本面因子特征提取完成！(批量去重名补丁已生效)")

✅ 10个量价与基本面因子特征提取完成！(批量去重名补丁已生效)


In [5]:
volume_df

,a,ag,al,ao,ap,au,b,bc,br,bu,...,ss,t,ta,tf,tl,ts,ur,v,y,zn
201601050000,44669,287111,131382,NaN,NaN,131198,NaN,NaN,NaN,411445,...,NaN,21604,590574,22416,NaN,NaN,NaN,10043,260505,77372
201601060000,69550,204526,101762,NaN,NaN,134834,NaN,NaN,NaN,329538,...,NaN,23014,432376,22715,NaN,NaN,NaN,7741,340569,191288
201601070000,53230,488186,116908,NaN,NaN,237981,NaN,NaN,NaN,439816,...,NaN,34218,473922,35229,NaN,NaN,NaN,14801,347021,227274
201601080000,52565,379737,146565,NaN,NaN,189270,NaN,NaN,NaN,478017,...,NaN,25004,497215,27056,NaN,NaN,NaN,8874,286702,287021
201601110000,72503,283036,177193,NaN,NaN,176926,NaN,NaN,NaN,310291,...,NaN,22898,425059,21535,NaN,NaN,NaN,11796,342777,206985
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202509240000,113327,753535,107916,308521.0,98360.0,285621,146261.0,3103.0,113405.0,152942,...,122330.0,91005,655697,60601,138791.0,32836.0,193224.0,902670,310807,109733
202509250000,171437,700711,146073,329000.0,59859.0,270576,132464.0,11240.0,83685.0,175377,...,129897.0,131434,701004,80972,160255.0,39540.0,95817.0,630324,361847,158280
202509260000,123642,957978,115180,280445.0,109757.0,270430,107807.0,5553.0,191754.0,210079,...,175317.0,81745,632930,60231,111562.0,32311.0,114516.0,734256,278742,126716
202509290000,112252,1527083,132925,356985.0,136145.0,333591,90643.0,7822.0,158160.0,152121,...,163271.0,85459,580757,68631,127403.0,26973.0,170093.0,654589,236751,180545


# 均值预测

In [6]:
from my_backtest import normalize_factor,compare_multi_lags_only_ic
import numpy as np
volume_target = np.log1p(volume_panel.fillna(method='ffill'))


In [7]:
import numpy as np
import pandas as pd
from IPython.display import display

def calculate_ma_benchmarks(true_df, windows=[1, 5, 10, 20]):
    """
    计算不同滚动均值窗口下的基准 MAE 和 RMSE
    注意：使用 shift(1) 确保不包含当天的未来数据
    """
    # 1. 强行清洗时间格式
    df = true_df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        idx_str = df.index.astype(str).str.replace(r'\.0$', '', regex=True).str[:8]
        df.index = pd.to_datetime(idx_str, format='%Y%m%d', errors='coerce')
    df = df.loc[df.index.notna()]
    
    results = []
    
    # 2. 循环计算不同窗口的均值误差
    for w in windows:
        # 使用过去 w 天的真实均值，作为今天的预测值
        ma_pred = df.shift(1).rolling(window=w, min_periods=1).mean()
        
        p_flat = ma_pred.values.flatten()
        t_flat = df.values.flatten()
        
        # 剔除空值对齐计算
        valid_mask = ~(np.isnan(p_flat) | np.isnan(t_flat))
        p_valid = p_flat[valid_mask]
        t_valid = t_flat[valid_mask]
        
        if len(p_valid) == 0:
            continue
            
        mae = np.mean(np.abs(p_valid - t_valid))
        rmse = np.sqrt(np.mean((p_valid - t_valid)**2))
        
        results.append({
            'Benchmark': f'MA-{w} Days (过去{w}天均值)',
            'MAE': mae,
            'RMSE': rmse
        })
        
    res_df = pd.DataFrame(results).set_index('Benchmark')
    return res_df

print("=== 正在计算纯均值基准 (Moving Average Benchmarks) 的预测误差 ===")
# 传入你清洗过的目标数据
ma_benchmarks = calculate_ma_benchmarks(volume_target, windows=[1, 5, 10, 20])

# 使用红黄绿色阶，颜色越绿代表误差越小、效果越好
display(ma_benchmarks.style.format("{:.4f}").background_gradient(cmap='RdYlGn_r'))

=== 正在计算纯均值基准 (Moving Average Benchmarks) 的预测误差 ===


,MAE,RMSE
Benchmark,,
MA-1 Days (过去1天均值),0.2484,0.3365
MA-5 Days (过去5天均值),0.2311,0.3113
MA-10 Days (过去10天均值),0.2462,0.3312
MA-20 Days (过去20天均值),0.2673,0.3622


In [8]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr

def evaluate_multiple_ma_multi_step(true_df, windows=[1, 5, 10, 20], horizons=[1, 2, 3, 4, 5, 6, 7]):
    """
    一键计算多种滚动均值（1天, 5天, 10天, 20天）对未来 1~7 步的跨期预测能力衰减
    """
    # 1. 强行清洗时间格式
    df = true_df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        idx_str = df.index.astype(str).str.replace(r'\.0$', '', regex=True).str[:8]
        df.index = pd.to_datetime(idx_str, format='%Y%m%d', errors='coerce')
    df = df.loc[df.index.notna()]
    
    results = []
    
    for w in windows:
        print(f"正在计算 MA-{w} (过去{w}天均值) 的 1~7 步预测误差...")
        # t 时刻我们能拿到的最新 N 天均值
        ma_at_t = df.rolling(window=w, min_periods=1).mean()
        
        for h in horizons:
            # t+h 时刻的真实值拉回 t 行对齐
            true_at_t_plus_h = df.shift(-h)
            
            p_flat = ma_at_t.values.flatten()
            t_flat = true_at_t_plus_h.values.flatten()
            
            valid_mask = ~(np.isnan(p_flat) | np.isnan(t_flat))
            p_valid = p_flat[valid_mask]
            t_valid = t_flat[valid_mask]
            
            if len(p_valid) == 0:
                continue
                
            mae = np.mean(np.abs(p_valid - t_valid))
            rmse = np.sqrt(np.mean((p_valid - t_valid)**2))
            
            # 计算截面 Rank IC
            ic_list = []
            for date in ma_at_t.index:
                p_s = ma_at_t.loc[date].dropna()
                t_s = true_at_t_plus_h.loc[date].dropna()
                idx = p_s.index.intersection(t_s.index)
                if len(idx) > 2:
                    r, _ = spearmanr(p_s[idx], t_s[idx])
                    ic_list.append(r)
            
            mean_ic = np.nanmean(ic_list)
            
            results.append({
                'Horizon': f't+{h}',
                'Window': f'MA-{w}',
                'Rank IC': mean_ic,
                'MAE': mae,
                'RMSE': rmse
            })
            
    return pd.DataFrame(results)

# =========================================================
# 调用计算
# =========================================================
multi_stats = evaluate_multiple_ma_multi_step(volume_target, windows=[1, 5, 10, 20])

# 将长表转化为三张透视表（宽表），方便横向与纵向比对
ic_df = multi_stats.pivot(index='Horizon', columns='Window', values='Rank IC')
mae_df = multi_stats.pivot(index='Horizon', columns='Window', values='MAE')
rmse_df = multi_stats.pivot(index='Horizon', columns='Window', values='RMSE')

# 确保列的顺序是从短到长排列
cols = [f'MA-{w}' for w in [1, 5, 10, 20]]
ic_df = ic_df[cols]
mae_df = mae_df[cols]
rmse_df = rmse_df[cols]

# =========================================================
# 绚丽展示
# =========================================================
print("\n" + "="*50)
print(" 1. Rank IC (截面排序相关性) 衰减对比 [越绿越好]")
print("="*50)
display(ic_df.style.format("{:.4f}").background_gradient(cmap='RdYlGn', axis=None))

print("\n" + "="*50)
print(" 2. MAE (平均绝对误差) 衰减对比 [越绿越好/数值越小越好]")
print("="*50)
display(mae_df.style.format("{:.4f}").background_gradient(cmap='RdYlGn_r', axis=None))

print("\n" + "="*50)
print(" 3. RMSE (均方根误差) 衰减对比 [越绿越好/数值越小越好]")
print("="*50)
display(rmse_df.style.format("{:.4f}").background_gradient(cmap='RdYlGn_r', axis=None))

正在计算 MA-1 (过去1天均值) 的 1~7 步预测误差...
正在计算 MA-5 (过去5天均值) 的 1~7 步预测误差...
正在计算 MA-10 (过去10天均值) 的 1~7 步预测误差...
正在计算 MA-20 (过去20天均值) 的 1~7 步预测误差...

 1. Rank IC (截面排序相关性) 衰减对比 [越绿越好]


Window,MA-1,MA-5,MA-10,MA-20
Horizon,,,,
t+1,0.9607,0.9664,0.9633,0.9579
t+2,0.9546,0.9621,0.9600,0.9555
t+3,0.9499,0.9586,0.9571,0.9533
t+4,0.9460,0.9555,0.9546,0.9513
t+5,0.9430,0.9528,0.9522,0.9494
t+6,0.9398,0.9503,0.9500,0.9476
t+7,0.9373,0.9480,0.9481,0.9460



 2. MAE (平均绝对误差) 衰减对比 [越绿越好/数值越小越好]


Window,MA-1,MA-5,MA-10,MA-20
Horizon,,,,
t+1,0.2484,0.2311,0.2462,0.2673
t+2,0.2744,0.2495,0.2595,0.2761
t+3,0.2909,0.2637,0.2704,0.2837
t+4,0.3033,0.2757,0.2800,0.2904
t+5,0.3134,0.2860,0.2884,0.2967
t+6,0.3265,0.2950,0.2959,0.3025
t+7,0.3357,0.3030,0.3026,0.3077



 3. RMSE (均方根误差) 衰减对比 [越绿越好/数值越小越好]


Window,MA-1,MA-5,MA-10,MA-20
Horizon,,,,
t+1,0.3365,0.3113,0.3312,0.3622
t+2,0.3708,0.3361,0.3498,0.3747
t+3,0.3928,0.3551,0.3649,0.3855
t+4,0.4098,0.3712,0.3783,0.3952
t+5,0.4233,0.3849,0.3901,0.4041
t+6,0.4392,0.3973,0.4007,0.4124
t+7,0.4506,0.4083,0.4100,0.4199


In [9]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

def compare_flat_vs_recursive_ma(true_df, window=5, horizons=[1, 2, 3, 4, 5, 6, 7]):
    """
    对比【平推法】与【递进法】在未来 1~7 步预测中的表现差异
    """
    # 1. 清洗时间格式
    df = true_df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        idx_str = df.index.astype(str).str.replace(r'\.0$', '', regex=True).str[:8]
        df.index = pd.to_datetime(idx_str, format='%Y%m%d', errors='coerce')
    df = df.loc[df.index.notna()]
    
    # ==========================================
    # 2. 生成预测矩阵
    # ==========================================
    # 方法A：平推 (Flat-line)
    flat_preds = {}
    ma_at_t = df.rolling(window=window, min_periods=1).mean()
    for h in horizons:
        flat_preds[h] = ma_at_t
        
    # 方法B：递进 (Recursive)
    recursive_preds = {}
    # 初始化 t 时刻的真实历史窗口 [t-4, t-3, t-2, t-1, t]
    hist_dfs = [df.shift(i) for i in range(window-1, -1, -1)]
    
    for h in horizons:
        # 递进计算：用最新的 5 个矩阵相加求平均
        pred_h = sum(hist_dfs[-window:]) / window
        recursive_preds[h] = pred_h
        # 将刚算出的预测值追加到历史列表中，供计算下一期使用！
        hist_dfs.append(pred_h) 

    # ==========================================
    # 3. 评估与统计
    # ==========================================
    results = []
    print("正在并行计算平推与递进两种方法的误差，请稍候...")
    
    for h in horizons:
        true_at_t_plus_h = df.shift(-h)
        
        for method_name, preds_dict in [('1_Flat-line (平推)', flat_preds), ('2_Recursive (递进)', recursive_preds)]:
            p_df = preds_dict[h]
            
            p_flat = p_df.values.flatten()
            t_flat = true_at_t_plus_h.values.flatten()
            
            valid_mask = ~(np.isnan(p_flat) | np.isnan(t_flat))
            p_valid = p_flat[valid_mask]
            t_valid = t_flat[valid_mask]
            
            if len(p_valid) == 0: continue
                
            rmse = np.sqrt(np.mean((p_valid - t_valid)**2))
            
            # 计算截面 Rank IC
            ic_list = []
            for date in p_df.index:
                p_s = p_df.loc[date].dropna()
                t_s = true_at_t_plus_h.loc[date].dropna()
                idx = p_s.index.intersection(t_s.index)
                if len(idx) > 2:
                    r, _ = spearmanr(p_s[idx], t_s[idx])
                    ic_list.append(r)
            mean_ic = np.nanmean(ic_list)
            
            results.append({
                'Horizon': f't+{h}',
                'Method': method_name,
                'Rank IC': mean_ic,
                'RMSE': rmse
            })
            
    return pd.DataFrame(results)

# =========================================================
# 调用计算并展示对比透视表
# =========================================================
compare_stats = compare_flat_vs_recursive_ma(volume_target, window=5)

ic_df = compare_stats.pivot(index='Horizon', columns='Method', values='Rank IC')
rmse_df = compare_stats.pivot(index='Horizon', columns='Method', values='RMSE')

print("\n" + "="*50)
print(" 🥇 Rank IC (截面排序相关性) 对比 [越高越好]")
print("="*50)
display(ic_df.style.format("{:.5f}").background_gradient(cmap='Greens', axis=None))

print("\n" + "="*50)
print(" 🎯 RMSE (绝对误差) 对比 [越低越好]")
print("="*50)
display(rmse_df.style.format("{:.5f}").background_gradient(cmap='Oranges', axis=None))

正在并行计算平推与递进两种方法的误差，请稍候...



 🥇 Rank IC (截面排序相关性) 对比 [越高越好]


Method,1_Flat-line (平推),2_Recursive (递进)
Horizon,,
t+1,0.96643,0.96644
t+2,0.96210,0.96242
t+3,0.95858,0.95893
t+4,0.95552,0.95594
t+5,0.95283,0.95341
t+6,0.95028,0.95073
t+7,0.94796,0.94828



 🎯 RMSE (绝对误差) 对比 [越低越好]


Method,1_Flat-line (平推),2_Recursive (递进)
Horizon,,
t+1,0.31127,0.31072
t+2,0.33610,0.33309
t+3,0.35510,0.35132
t+4,0.37119,0.36669
t+5,0.38495,0.37993
t+6,0.39732,0.39411
t+7,0.40835,0.40576


# 导出

In [10]:
import pandas as pd
import numpy as np
import pickle
import os

def generate_mpo_volume_recursive_ma(true_df, window=5, horizons=[1, 2, 3, 4, 5, 6, 7]):
    """
    ??????? MA5 ??? (Recursive MA) ? 7 ????
    ?????? MPO ????? {(t, tau): Series} ?????
    ??? tau ??????????????
    """
    print("1. ????????...")
    df = true_df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        idx_str = df.index.astype(str).str.replace(r'\.0$', '', regex=True).str[:8]
        df.index = pd.to_datetime(idx_str, format='%Y%m%d', errors='coerce')
    df = df.loc[df.index.notna()].sort_index()

    print(f"2. ???? MA-{window} ? 1~7 ??????? (Recursive)...")
    recursive_preds = {}

    # ??? t ????????? [t-4, t-3, t-2, t-1, t]
    hist_dfs = [df.shift(i) for i in range(window - 1, -1, -1)]

    for h in horizons:
        pred_h = sum(hist_dfs[-window:]) / window
        recursive_preds[h] = pred_h
        hist_dfs.append(pred_h)

    print("3. ????? MPO ?????? {(t, tau): Series} ...")
    mpo_volume_data = {}
    trading_index = pd.Index(df.index.unique()).sort_values()
    pos_map = {pd.Timestamp(ts): i for i, ts in enumerate(trading_index)}

    for t in trading_index:
        pos = pos_map[pd.Timestamp(t)]
        for h in horizons:
            future_pos = pos + int(h)
            if future_pos >= len(trading_index):
                continue
            tau = pd.Timestamp(trading_index[future_pos])
            pred_series = recursive_preds[h].loc[t].fillna(0)
            mpo_volume_data[(pd.Timestamp(t), tau)] = pred_series

    return mpo_volume_data

# =========================================================
# 1. ??????? (????? volume_target)
# =========================================================
mpo_volume_dict = generate_mpo_volume_recursive_ma(volume_target, window=5)

# =========================================================
# 2. ??????????? (????? volume data ???????)
# =========================================================
output_dir = r"C:\Users\CtaBackup\Desktop\opt\data"
output_path = os.path.join(output_dir, "mpo_volume_data.pkl")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# =========================================================
# 3. ??? Pickle ??
# =========================================================
with open(output_path, "wb") as f:
    pickle.dump(mpo_volume_dict, f)

print(f"\n? ???????")
print(f"????? {len(mpo_volume_dict)} ? (t, tau) ????")
print(f"??????: {output_path}")



1. ????????...
2. ???? MA-5 ? 1~7 ??????? (Recursive)...
3. ????? MPO ?????? {(t, tau): Series} ...

? ???????
????? 16555 ? (t, tau) ????
??????: C:\Users\CtaBackup\Desktop\opt\data\mpo_volume_data.pkl
